# Agente Autônomo de EDA - execução segura no Google Colab

Este notebook executa a versão atual do projeto diretamente do repositório público:

- **GitHub:** https://github.com/AdryRocha/agente-eda-streamlit
- **Aplicação pública:** https://agente-eda-app-adryrocha25.streamlit.app/

## Segurança

Antes de executar, abra o painel **Secrets** do Colab (ícone de chave) e cadastre:

- `GEMINI_API_KEY`
- `NGROK_AUTH_TOKEN`

Ative o acesso desses dois Secrets ao notebook. As credenciais não ficam gravadas nas células nem no arquivo `.ipynb`.

> O notebook cria um arquivo `.streamlit/secrets.toml` apenas na máquina virtual temporária do Colab. Esse arquivo não é enviado ao GitHub e desaparece quando a sessão é encerrada.

In [ ]:
# 1. Clonar a versão atual e instalar as dependências
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/AdryRocha/agente-eda-streamlit.git"
REPO_DIR = Path("/content/agente-eda-streamlit")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pyngrok", "requests"],
    check=True,
)

os.chdir(REPO_DIR)
print("Projeto clonado e dependências instaladas com sucesso.")

## Configurar as credenciais

A célula abaixo lê os valores do painel Secrets, valida sua existência e cria a configuração temporária necessária ao Streamlit.

Nenhuma credencial será impressa na tela.

In [ ]:
# 2. Ler Secrets e criar configuração temporária do Streamlit
from google.colab import userdata
from pathlib import Path
import json


def read_colab_secret(name: str) -> str:
    """Lê um Secret do Colab e apresenta uma mensagem clara se ele não existir."""
    try:
        value = userdata.get(name)
    except Exception as exc:
        raise ValueError(
            f"Cadastre {name} no painel Secrets do Colab e habilite o acesso ao notebook."
        ) from exc

    if not value or not str(value).strip():
        raise ValueError(
            f"O Secret {name} está vazio. Informe um valor válido no painel Secrets do Colab."
        )

    return str(value).strip()


GEMINI_API_KEY = read_colab_secret("GEMINI_API_KEY")
NGROK_AUTH_TOKEN = read_colab_secret("NGROK_AUTH_TOKEN")

streamlit_dir = Path(".streamlit")
streamlit_dir.mkdir(exist_ok=True)

# json.dumps produz uma string entre aspas compatível com o formato TOML.
(streamlit_dir / "secrets.toml").write_text(
    f"GEMINI_API_KEY = {json.dumps(GEMINI_API_KEY)}\n",
    encoding="utf-8",
)

print("Credenciais carregadas com segurança e configuração temporária criada.")

## Iniciar a aplicação

A célula seguinte encerra processos antigos, inicia o Streamlit em segundo plano e aguarda a porta 8501 ficar disponível.

In [ ]:
# 3. Iniciar o Streamlit em segundo plano
import os
import socket
import subprocess
import time
from pathlib import Path

# Encerra uma execução anterior do Streamlit, se existir.
subprocess.run(["pkill", "-f", "streamlit run app.py"], check=False)

log_path = Path("/content/streamlit.log")
log_file = log_path.open("w", encoding="utf-8")

streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    cwd=str(REPO_DIR),
)

def port_is_open(host="127.0.0.1", port=8501):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0

for _ in range(60):
    if streamlit_process.poll() is not None:
        log_file.close()
        raise RuntimeError(f"O Streamlit encerrou antes de iniciar. Consulte {log_path}.")
    if port_is_open():
        break
    time.sleep(1)
else:
    raise TimeoutError("O Streamlit não iniciou na porta 8501 dentro do tempo esperado.")

print("Streamlit iniciado com sucesso na porta 8501.")

## Criar o link público temporário

O endereço do ngrok é válido apenas enquanto esta sessão do Colab estiver ativa.

In [ ]:
# 4. Abrir o túnel do ngrok
from pyngrok import ngrok

# Fecha túneis antigos desta sessão para evitar duplicidade.
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_tunnel = ngrok.connect(8501, "http")
public_url = public_tunnel.public_url

print("Aplicação disponível temporariamente em:")
print(public_url)

## Encerrar a execução

Execute a célula abaixo quando terminar. Ela fecha o túnel, encerra o Streamlit e remove o arquivo temporário de Secrets.

In [ ]:
# 5. Encerrar a aplicação e limpar a sessão
from pathlib import Path
from pyngrok import ngrok

ngrok.kill()

if "streamlit_process" in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    try:
        streamlit_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        streamlit_process.kill()

if "log_file" in globals() and not log_file.closed:
    log_file.close()

secrets_file = Path(REPO_DIR) / ".streamlit" / "secrets.toml"
if secrets_file.exists():
    secrets_file.unlink()

print("Túnel encerrado, Streamlit finalizado e Secret temporário removido.")